# Taller 1 - Fase 3: Ingeniería de Prompts para EcoMarket

Juan David Cruz Garcia - A00368421

**Maestría en Inteligencia Artificial Aplicada**  
**Materia:** Inteligencia Artificial Generativa  

---

### Estrategias de Prompting Aplicadas en este Notebook:
1. **Definición de Sistema (Rol y Restricciones):** Se establece la identidad del agente y límites de comportamiento estrictos.
2. **Few-Shot Prompting:** Se incorporan 2 ejemplos dorados (*golden examples*) por ejercicio para guiar el tono y estructura de respuesta.
3. **Delimitadores (Delimiters):** Se utilizan marcas explícitas (`--- CONTEXTO ---`) para separar las instrucciones del contenido inyectado.
4. **Chain of Thought (CoT):** Guía paso a paso en el *System Prompt* para estructurar el razonamiento del modelo.
5. **Inyección de Contexto (Simulación RAG):** Datos reales recuperados de la base de datos de EcoMarket inyectados en tiempo de inferencia.
6. **Control de Temperatura:** Temperatura baja (`0.3`) para garantizar consistencia y precisión factual transaccional.

---
### Opciones de Ejecución sin Costo / API Keys:
- **Opción A (Groq API - 100% Gratuita):** Puedes crear una API Key gratuita en [console.groq.com](https://console.groq.com) sin tarjeta de crédito y usar modelos como `llama-3.1-8b-instant` con la misma librería de OpenAI.
- **Opción B (OpenAI GPT-4o-mini / GPT-3.5-turbo):** Si tienes créditos de OpenAI, la ejecución de todo este Notebook cuesta menos de **$0.001 USD**.
- **Opción C (Modo Simulación Offline - $0 USD):** Si no ingresas ninguna API key, el notebook ejecuta un simulador local que muestra los prompts orquestados listos para validación.

## 1. Importación de Librerías y Configuración del Cliente API

In [2]:
pip install openai

  Using cached openai-3.8.0-py3-none-any.whl.metadata (41 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiter-0.16.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
Using cached openai-3.8.0-py3-none-any.whl (1.7 MB)
Using cached httpx2-2.12.0-py3-none-any.whl (95 kB)
Using cached httpcore2-2.12.0-py3-none-any.whl (83 kB)
Using cached jiter-0.16.0-cp312-cp312-win_amd64.whl (196 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 14.2 MB/s  0:00:00
Using cached truststore-0.10.4-py3-none-any.whl (18 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ----------------------------------------  0/10 [typing-inspection]
   ---- -----------------------------------  1/10 [truststore]

In [ ]:
import os
from openai import OpenAI

# Pega aquí tu API Key (o desde tu variable de entorno)
API_KEY = os.environ.get("OPENAI_API_KEY", "") or os.environ.get("GROQ_API_KEY", "")


# Si estás usando la URL base de OpenRouter u otro gateway:
# BASE_URL = "https://openrouter.ai/api/v1"  # Descomenta si usas OpenRouter
BASE_URL = "https://api.groq.com/openai/v1"

def obtener_cliente_llm():
    if API_KEY:
        # Usamos el modelo que tienes seleccionado en tu pantalla
        MODELO_SELECCIONADO = "openai/gpt-oss-120b"
        print(f" Conectado exitosamente. Usando modelo: {MODELO_SELECCIONADO}")
        return OpenAI(base_url=BASE_URL, api_key=API_KEY), MODELO_SELECCIONADO
    else:
        print("ℹ️ [MODO SIMULACIÓN OFFLINE - SIN API KEY]")
        return None, None

client, MODEL_NAME = obtener_cliente_llm()

 Conectado exitosamente. Usando modelo: openai/gpt-oss-120b


## 2. Base de Datos Ficticia y Capa de Recuperación (RAG)

In [14]:
# Base de datos transaccional simulada de EcoMarket (10 pedidos + catálogo)
ECOMARKET_DB = {
    "pedidos": {
        "ORD-001": {"cliente": "Ana M.", "producto": "Termo de Acero Inoxidable", "estado": "En tránsito", "fecha_estimada": "2023-10-25", "retrasado": False},
        "ORD-002": {"cliente": "Carlos R.", "producto": "Bolsa Reutilizable Pack x3", "estado": "Retrasado en aduana", "fecha_estimada": "2023-11-02", "retrasado": True},
        "ORD-003": {"cliente": "María L.", "producto": "Kit de Pajillas de Bambú", "estado": "Entregado", "fecha_estimada": "2023-10-15", "retrasado": False},
        "ORD-004": {"cliente": "Diego S.", "producto": "Jabón Artesanal Orgánico", "estado": "Procesando", "fecha_estimada": "2023-10-30", "retrasado": False},
        "ORD-005": {"cliente": "Laura P.", "producto": "Cepillo de Dientes de Bambú", "estado": "Cancelado por el cliente", "fecha_estimada": "N/A", "retrasado": False},
        "ORD-006": {"cliente": "Juan D.", "producto": "Botella Plegable de Silicona", "estado": "En tránsito", "fecha_estimada": "2023-10-26", "retrasado": False},
        "ORD-007": {"cliente": "Sofía V.", "producto": "Set de Cubiertos Portátiles", "estado": "Extraviado por transportista", "fecha_estimada": "N/A", "retrasado": True},
        "ORD-008": {"cliente": "Andrés G.", "producto": "Champú Sólido Natural", "estado": "Entregado", "fecha_estimada": "2023-10-10", "retrasado": False},
        "ORD-009": {"cliente": "Valentina C.", "producto": "Envoltura de Cera de Abeja", "estado": "Devuelto al remitente", "fecha_estimada": "N/A", "retrasado": True},
        "ORD-010": {"cliente": "Felipe M.", "producto": "Esponja Vegetal de Luffa", "estado": "En tránsito", "fecha_estimada": "2023-10-28", "retrasado": False},
    },
    "catalogo": {
        "PROD-001": {"nombre": "Cepillo de Dientes de Bambú", "categoria": "Higiene Personal", "devolucion_permitida": False, "motivo_restriccion": "Producto de higiene personal de uso íntimo."},
        "PROD-002": {"nombre": "Termo de Acero Inoxidable", "categoria": "Accesorios Reutilizables", "devolucion_permitida": True, "motivo_restriccion": None},
        "PROD-003": {"nombre": "Jabón Artesanal Orgánico", "categoria": "Higiene Personal", "devolucion_permitida": False, "motivo_restriccion": "Producto perecedero de base orgánica con vida útil limitada."},
        "PROD-004": {"nombre": "Bolsa Reutilizable Pack x3", "categoria": "Accesorios Reutilizables", "devolucion_permitida": True, "motivo_restriccion": None},
        "PROD-005": {"nombre": "Kit de Pajillas de Bambú", "categoria": "Accesorios Reutilizables", "devolucion_permitida": True, "motivo_restriccion": None},
    }
}

def recuperar_contexto_pedido(tracking_number: str) -> str:
    pedido = ECOMARKET_DB["pedidos"].get(tracking_number)
    if not pedido:
        return f"[SIN RESULTADOS] No existe un pedido con el número {tracking_number} en el sistema."
    return (
        f"Número de pedido: {tracking_number}\n"
        f"Cliente: {pedido['cliente']}\n"
        f"Producto: {pedido['producto']}\n"
        f"Estado logístico actual: {pedido['estado']}\n"
        f"Fecha estimada de entrega: {pedido['fecha_estimada']}\n"
        f"Presenta retraso: {'Sí' if pedido['retrasado'] else 'No'}\n"
        f"Enlace de rastreo: https://ecomarket.com/track/{tracking_number}"
    )

def recuperar_contexto_producto(producto_id: str) -> str:
    producto = ECOMARKET_DB["catalogo"].get(producto_id)
    if not producto:
        return f"[SIN RESULTADOS] No existe un producto con el ID {producto_id} en el catálogo."
    restriccion = producto['motivo_restriccion'] if producto['motivo_restriccion'] else "Sin restricciones."
    return (
        f"ID del producto: {producto_id}\n"
        f"Nombre: {producto['nombre']}\n"
        f"Categoría: {producto['categoria']}\n"
        f"Devolución permitida: {'Sí' if producto['devolucion_permitida'] else 'No'}\n"
        f"Motivo de restricción: {restriccion}"
    )

def generar_respuesta(system_prompt: str, user_prompt: str, temperatura: float = 0.3):
    if client is None:
        print("\n--- PROMPT ORQUESTADO PARA VALIDACIÓN ---")
        print(f"SYSTEM PROMPT:\n{system_prompt}\n")
        print(f"USER PROMPT:\n{user_prompt}")
        return None
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperatura,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

## 3. Ejercicio 1: Prompt de Solicitud de Estado de Pedido

**Estrategia:** System Role + Chain of Thought + Few-Shot (2 ejemplos) + Delimiters + Context Injection.

In [15]:
system_prompt_pedido = """Eres un agente de servicio al cliente amable y profesional de EcoMarket, una tienda en línea de productos sostenibles.

--- INSTRUCCIONES PASO A PASO ---
1. Lee cuidadosamente los datos del pedido proporcionados en la sección de CONTEXTO.
2. Saluda al cliente por su nombre si está disponible en el contexto.
3. Comunica el estado actual del pedido de forma clara y directa.
4. Si el pedido presenta retraso o está extraviado, muestra empatía genuina, ofrece una disculpa y explica que el equipo de logística está trabajando en la resolución.
5. Incluye la fecha estimada de entrega (si aplica) y el enlace de rastreo.
6. Si el pedido fue cancelado o devuelto, explica brevemente el estado sin inventar razones.

--- RESTRICCIONES ---
- Responde ÚNICAMENTE con la información presente en el CONTEXTO. No inventes fechas, políticas ni estados logísticos.
- Si el pedido no se encuentra en el sistema, indícalo honestamente y ofrece al cliente comunicarse con un agente humano.
- No solicites datos sensibles como tarjeta de crédito o contraseñas.
- Mantén la respuesta concisa: máximo 4 oraciones.

--- EJEMPLOS ---

Ejemplo 1 (pedido normal):
Cliente pregunta: "¿Ya salió mi pedido ORD-006?"
Contexto del sistema: Estado: En tránsito | Fecha estimada: 2023-10-26 | Retrasado: No
Respuesta ideal: "¡Hola, Juan! Tu pedido ORD-006 ya está en camino. La fecha estimada de entrega es el 26 de octubre. Puedes seguir el estado en tiempo real aquí: https://ecomarket.com/track/ORD-006. ¡Gracias por elegirnos!"

Ejemplo 2 (pedido con problema):
Cliente pregunta: "Mi pedido ORD-007 no ha llegado, ¿qué pasó?"
Contexto del sistema: Estado: Extraviado por transportista | Fecha estimada: N/A | Retrasado: Sí
Respuesta ideal: "Hola, Sofía. Lamento mucho informarte que tu pedido ORD-007 fue reportado como extraviado por la empresa transportista. Nuestro equipo de logística ya abrió una investigación para localizarlo y resolverlo lo antes posible. Te mantendremos informada por correo electrónico. Disculpa las molestias."
"""

def probar_solicitud_pedido(tracking_number):
    print(f"\n{'='*60}\nPROBANDO PEDIDO: {tracking_number}\n{'='*60}")
    contexto = recuperar_contexto_pedido(tracking_number)
    
    user_prompt = f"""--- CONTEXTO RECUPERADO DE LA BASE DE DATOS ---
{contexto}
--- FIN DEL CONTEXTO ---

Mensaje del cliente: "Hola, estoy preocupado. Necesito saber qué pasó con mi pedido {tracking_number}, no ha llegado."
"""
    
    respuesta = generar_respuesta(system_prompt_pedido, user_prompt)
    if respuesta:
        print("\n🤖 RESPUESTA GENERADA:\n")
        print(respuesta)

# Prueba con pedido retrasado (ORD-002) y pedido normal (ORD-006)
probar_solicitud_pedido("ORD-002")
probar_solicitud_pedido("ORD-006")


PROBANDO PEDIDO: ORD-002

🤖 RESPUESTA GENERADA:

Hola, Carlos R. Lamento mucho la demora; su pedido ORD-002 está retrasado en la aduana. La fecha estimada de entrega es el 2 de noviembre y puede seguirlo aquí: https://ecomarket.com/track/ORD-002. Estamos trabajando para resolverlo lo antes posible.

PROBANDO PEDIDO: ORD-006

🤖 RESPUESTA GENERADA:

¡Hola, Juan! Tu pedido ORD-006 está en tránsito y se espera que llegue el 26 de octubre. Puedes seguirlo aquí: https://ecomarket.com/track/ORD-006. Gracias por tu paciencia.


## 4. Ejercicio 2: Prompt de Devolución de Producto

**Estrategia:** Condicionales de Negocio + Empatía + Few-Shot (Aprobado vs Rechazado) + Delimiters.

In [16]:
system_prompt_devolucion = """Eres un representante de soporte de EcoMarket, empático y resolutivo. Tu tarea es guiar a los clientes en el proceso de devolución de productos.

--- INSTRUCCIONES PASO A PASO (CHAIN OF THOUGHT) ---
1. Lee la ficha del producto en la sección de CONTEXTO.
2. Determina si la devolución está permitida revisando el campo "Devolución permitida".
3. Si la devolución SÍ está permitida:
   a. Explica amablemente el proceso: el cliente debe ingresar a ecomarket.com/devoluciones, generar su etiqueta de envío gratuita e ir al punto de recolección más cercano.
   b. Indica que el reembolso se procesa en 5-7 días hábiles tras recibir el producto.
4. Si la devolución NO está permitida:
   a. Explica con empatía el motivo de la restricción (higiene, producto perecedero, etc.) citando el motivo que aparece en el CONTEXTO.
   b. Ofrece proactivamente un código de descuento del 15% (código: ECOCARE15) para su próxima compra como gesto de buena voluntad.
5. Cierra con un tono cálido y abierto a más preguntas.

--- RESTRICCIONES ---
- NUNCA apruebes una devolución que el sistema marque como no permitida. Esto es una regla de negocio firme.
- No inventes motivos de restricción que no estén en el CONTEXTO.
- Mantén la respuesta concisa y directa: máximo 5 oraciones.

--- EJEMPLOS ---

Ejemplo 1 (devolución PERMITIDA):
Cliente pregunta: "Quiero devolver las bolsas reutilizables PROD-004."
Contexto: Devolución permitida: Sí | Categoría: Accesorios Reutilizables
Respuesta ideal: "¡Claro que sí! Las bolsas reutilizables pueden devolverse dentro de los 30 días posteriores a la compra. Ingresa a ecomarket.com/devoluciones con tu número de pedido para generar tu etiqueta de envío gratuita. Una vez recibamos el producto, procesaremos tu reembolso en 5 a 7 días hábiles. ¿Hay algo más en lo que pueda ayudarte?"

Ejemplo 2 (devolución NO permitida):
Cliente pregunta: "Compré el jabón artesanal PROD-003 pero no me gustó el aroma, ¿lo puedo devolver?"
Contexto: Devolución permitida: No | Motivo: Producto perecedero de base orgánica con vida útil limitada.
Respuesta ideal: "Entiendo tu situación y lamento que el aroma no haya sido de tu agrado. Desafortunadamente, el Jabón Artesanal Orgánico no puede ser devuelto ya que es un producto perecedero de base orgánica con vida útil limitada, y por políticas sanitarias no podemos reingresarlo al inventario. Sin embargo, como gesto de disculpa, te ofrezco un 15% de descuento en tu próxima compra con el código ECOCARE15. Esperamos que lo disfrutes."
"""

def probar_devolucion(producto_id):
    print(f"\n{'='*60}\nPROBANDO DEVOLUCIÓN PARA PRODUCTO: {producto_id}\n{'='*60}")
    contexto = recuperar_contexto_producto(producto_id)
    
    user_prompt = f"""--- CONTEXTO DEL CATÁLOGO DE PRODUCTOS ---
{contexto}
--- FIN DEL CONTEXTO ---

Mensaje del cliente: "Buen día, compré el producto {producto_id} pero ya no lo necesito. ¿Puedo hacer la devolución? ¿Cómo es el proceso?"
"""
    
    respuesta = generar_respuesta(system_prompt_devolucion, user_prompt)
    if respuesta:
        print("\n🤖 RESPUESTA GENERADA:\n")
        print(respuesta)

# Prueba con producto NO retornable (PROD-001 - Higiene) y SÍ retornable (PROD-002 - Termo)
probar_devolucion("PROD-001")
probar_devolucion("PROD-002")


PROBANDO DEVOLUCIÓN PARA PRODUCTO: PROD-001

🤖 RESPUESTA GENERADA:

Entiendo tu situación y lamento que ya no necesites el Cepillo de Dientes de Bambú. Desafortunadamente, no podemos aceptar la devolución porque es un producto de higiene personal de uso íntimo, y por normas sanitarias no podemos reingresarlo al inventario. Como gesto de buena voluntad, te ofrezco un 15 % de descuento en tu próxima compra con el código ECOCARE15. Si tienes alguna otra consulta, estaré encantado de ayudarte.

PROBANDO DEVOLUCIÓN PARA PRODUCTO: PROD-002

🤖 RESPUESTA GENERADA:

¡Claro que sí! Puedes devolver el Termo de Acero Inoxidable ingresando a ecomarket.com/devoluciones y generando tu etiqueta de envío gratuita. Lleva el paquete al punto de recolección más cercano que te indique la plataforma. Una vez recibamos el producto, procesaremos el reembolso en 5‑7 días hábiles. ¿Hay algo más en lo que pueda ayudarte?
